# 1 · Convert the raw Excel exports

Reads `data/raw/series_<n>.xlsx` and writes one CSV per sausage to `data/converted/`
(`series_<n>_sausage_<k>.csv`). Each file holds the uplinks of that sausage's two probes
(LSE01, LTC-FS) plus the shared WS202 uplinks, in arrival order: the stream the models see.

* Series 1 is annotated (column `label`: Umroetung, Reifung Phase 1, Reifung Phase 2). Series 2 is not.
* Dropped: pH probes and the ambient temperature/humidity device. They are not in the demonstrator case.
* The logic lives in `ml_core/conversion.py` (tested in `tests/test_ml_core.py`).

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))  # project root, so ml_core is importable
import matplotlib.pyplot as plt
import pandas as pd
from ml_core import data
from ml_core.conversion import convert_all

In [ ]:
RAW_DIR = data.RAW_DIR
CONVERTED_DIR = data.CONVERTED_DIR

In [ ]:
summary = convert_all(RAW_DIR, CONVERTED_DIR)
summary

## Sanity check: one annotated sausage over time

In [ ]:
files = sorted(f for f in os.listdir(CONVERTED_DIR) if f.endswith(".csv"))
df = pd.read_csv(os.path.join(CONVERTED_DIR, files[0]), parse_dates=["timestamp"])
lse = df[df["sensor"] == "lse01"]
fig, ax = plt.subplots(figsize=(11, 3.5))
for label, part in lse.groupby(lse.get("label", pd.Series("unlabeled", index=lse.index)).fillna("unlabeled")):
    ax.scatter(part["timestamp"], part["lse01_moisture"], s=2, label=label)
ax.set(title=files[0], ylabel="LSE01 moisture (%)")
ax.legend(markerscale=6)
plt.show()